# Orders dataset cleaning 

In [ ]:
import pandas as pd
import numpy as np

In [ ]:


orders = pd.read_csv(r"C:\Users\MAINAK\Desktop\Data-Science\projects\MarketLens\data\raw\olist_orders_dataset.csv")

### Step 1 — Load & verify

- Shape
- Columns
- Data types
- Null values
- Duplicate rows

In [ ]:
orders.shape

In [ ]:
orders.columns

In [ ]:
orders.dtypes

In [ ]:
orders.isnull().sum()

In [ ]:
orders.duplicated().sum()

### Step 2 — Nulls vs Order Status

In [ ]:
orders[orders["order_approved_at"].isnull()]["order_status"].value_counts()

In [ ]:
orders[orders["order_delivered_carrier_date"].isnull()]["order_status"].value_counts()

In [ ]:
orders[orders["order_delivered_customer_date"].isnull()]["order_status"].value_counts()

In [ ]:
orders[
    (orders["order_approved_at"].isnull()) &
    (orders["order_status"] == "delivered")
]

In [ ]:
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])
orders["order_delivered_carrier_date"] = pd.to_datetime(orders["order_delivered_carrier_date"])

In [ ]:
orders[
    (orders["order_approved_at"].isnull()) &
    (orders["order_status"] == "delivered")
][["order_purchase_timestamp", "order_delivered_carrier_date"]]


In [ ]:
purchase_carrier_days = orders["order_delivered_carrier_date"] - orders["order_purchase_timestamp"]


In [ ]:
purchase_carrier_days.max()

### Date Columns Datetime Conversion

In [ ]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])

In [ ]:
orders.dtypes

In [ ]:
invalid_delivery = orders[orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"]]
len(invalid_delivery)

In [ ]:
invalid_carrier = orders[orders["order_delivered_carrier_date"] < orders["order_purchase_timestamp"]]
len(invalid_carrier)

In [ ]:
orders["has_invalid_timeline"] = (
    (orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"]) |
    (orders["order_delivered_carrier_date"] < orders["order_purchase_timestamp"])
)

orders["has_invalid_timeline"].value_counts()

In [ ]:
orders["order_status"].value_counts()

In [ ]:
orders.columns

In [ ]:
import os

os.makedirs("../data/processed", exist_ok=True)
orders.to_csv("../data/processed/orders_cleaned.csv", index=False)

## Order_Items data set Cleaning

In [ ]:
order_items = pd.read_csv(r"C:\Users\MAINAK\Desktop\Data-Science\projects\MarketLens\data\raw\olist_order_items_dataset.csv")

#  shape and columns 

print("Shape : ",order_items.shape)
print("\n Columns : ",order_items.columns.tolist())


In [ ]:
#. Check missing Values 
print("\nCheck missing values : \n",order_items.isnull().sum())

# exact duplicates row Check 

print("\n Duplicates rows : \n ", order_items.duplicated().sum())

In [ ]:
# Date _times change 
order_items["shipping_limit_date"] = pd.to_datetime(order_items["shipping_limit_date"])

# Statistical summary of numerical columns
order_items[["price", "freight_value"]].describe()

In [ ]:
print("Prices <= 0:", (order_items["price"] <= 0).sum())
print("Freight < 0:", (order_items["freight_value"] < 0).sum())
print("Freight == 0 (Free shipping):", (order_items["freight_value"] == 0).sum())

In [ ]:
# 1. Composite key uniqueness check
composite_duplicates = order_items.duplicated(subset=["order_id", "order_item_id"]).sum()
print("Composite key duplicates (order_id + order_item_id):", composite_duplicates)

# 2. Referential integrity: Orphan items check against orders
orphan_items = (~order_items["order_id"].isin(orders["order_id"])).sum()
print("Orphan items count (orders me missing):", orphan_items)

In [ ]:
# ensure processed directory exists
os.makedirs("../data/processed", exist_ok=True)

# save cleaned order_items
order_items.to_csv("../data/processed/order_items_cleaned.csv", index=False)

## Order_payments

In [ ]:
# 1. Load dataset
payments = pd.read_csv(r"C:\Users\MAINAK\Desktop\Data-Science\projects\MarketLens\data\raw\olist_order_payments_dataset.csv")

# 2. Basic info and shape
print("Shape:", payments.shape)
print("\nColumns:", payments.columns.tolist())
payments.info()

# 3. Missing values & Duplicates check
print("\nMissing values:\n", payments.isnull().sum())
print("\nExact duplicate rows:", payments.duplicated().sum())

In [ ]:
# Payment types distribution
print("Payment Types:\n", payments["payment_type"].value_counts())

In [ ]:
# 1. Summary statistics
print(payments[["payment_installments", "payment_value"]].describe())

# 2. Potential anomalies count
print("\nZero or Negative Payments:", (payments["payment_value"] <= 0).sum())
print("Zero Installments:", (payments["payment_installments"] == 0).sum())

In [ ]:
# 1. Inspect 'not_defined' payment types
print("--- NOT DEFINED ROWS ---")
display(payments[payments["payment_type"] == "not_defined"])

# 2. Inspect zero payment values
print("\n--- ZERO PAYMENT VALUE ROWS ---")
display(payments[payments["payment_value"] == 0])

# 3. Inspect zero installments
print("\n--- ZERO INSTALLMENTS ROWS ---")
display(payments[payments["payment_installments"] == 0])

In [ ]:
# Zero-payment order IDs ka status check
zero_pay_orders = payments[payments["payment_value"] == 0]["order_id"]
orders[orders["order_id"].isin(zero_pay_orders)][["order_id", "order_status"]]

In [ ]:
# 1. Drop 3 undefined, zero-value phantom records
payments_cleaned = payments[payments["payment_type"] != "not_defined"].copy()

# 2. Impute 0 installments to 1 (business rule for credit card transactions)
payments_cleaned.loc[payments_cleaned["payment_installments"] == 0, "payment_installments"] = 1

# 3. Verification checks
print("Remaining rows:", payments_cleaned.shape[0])
print("Zero installments left:", (payments_cleaned["payment_installments"] == 0).sum())
print("Not-defined count:", (payments_cleaned["payment_type"] == "not_defined").sum())

In [ ]:
# Check for orphan orders in payments
orphan_payments = (~payments_cleaned["order_id"].isin(orders["order_id"])).sum()
print("Orphan payment records:", orphan_payments)

In [ ]:
os.makedirs("../data/processed", exist_ok=True)
payments_cleaned.to_csv("../data/processed/order_payments_cleaned.csv", index=False)
print("Saved successfully to data/processed/order_payments_cleaned.csv")

## Products _ Data sets

In [ ]:
# Load products dataset
products = pd.read_csv("../data/raw/olist_products_dataset.csv")

# 1. Structure aur column names
print("Shape:", products.shape)
print("\nColumns:", products.columns.tolist())

# 2. Missing values check
print("\nMissing values:\n", products.isnull().sum())

# 3. Duplicate check
print("\nDuplicate rows:", products.duplicated().sum())
print("Duplicate product_ids:", products["product_id"].duplicated().sum())

In [ ]:
# Check overlap between missing category and missing dimensions
missing_cat = products["product_category_name"].isnull()
missing_dim = products["product_weight_g"].isnull()

print("Category missing count:", missing_cat.sum())
print("Dimensions missing count:", missing_dim.sum())
print("Overlap (Both missing):", (missing_cat & missing_dim).sum())

In [ ]:
# Load category translations
translations = pd.read_csv("../data/raw/product_category_name_translation.csv")

print("Translations Shape:", translations.shape)
print("\nSample Mappings:")
display(translations.head(10))

# Check missing categories in translation file
missing_in_trans = set(products["product_category_name"].dropna()) - set(translations["product_category_name"])
print("\nCategories in products but missing in translation table:", len(missing_in_trans))
print("Missing categories list:", missing_in_trans)

In [ ]:
# 1. Jo 2 missing categories thi, unhe translation table me add kar rahe hain
extra_translations = pd.DataFrame([
    {"product_category_name": "pc_gamer", "product_category_name_english": "pc_gamer"},
    {"product_category_name": "portateis_cozinha_e_preparadores_de_alimentos", "product_category_name_english": "portable_kitchen_food_preparers"}
])

translations_updated = pd.concat([translations, extra_translations], ignore_index=True)

# 2. Products table ke saath English translation merge kar rahe hain
products_cleaned = products.merge(translations_updated, on="product_category_name", how="left")

# 3. Missing categories (610) ko 'unknown' label de rahe hain
products_cleaned["product_category_name_english"] = products_cleaned["product_category_name_english"].fillna("unknown")
products_cleaned["product_category_name"] = products_cleaned["product_category_name"].fillna("unknown")

# 4. Check karte hain ki kya abhi bhi koi category missing hai
print("Shape before:", products.shape)
print("Shape after merge:", products_cleaned.shape)
print("\nMissing English Categories:", products_cleaned["product_category_name_english"].isnull().sum())
print("Missing Portuguese Categories:", products_cleaned["product_category_name"].isnull().sum())

In [ ]:
# Check remaining nulls in products_cleaned
print(products_cleaned.isnull().sum())


In [ ]:
# Check karte hain ki kya ye missing dimensions wale products order_items mein hain
missing_dim_ids = products_cleaned[products_cleaned["product_weight_g"].isnull()]["product_id"]
print("Missing dimension products ordered count:")
print(order_items[order_items["product_id"].isin(missing_dim_ids)].shape[0])

In [ ]:
# 1. Fill physical dimensions with median values
dimension_cols = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]
for col in dimension_cols:
    products_cleaned[col] = products_cleaned[col].fillna(products_cleaned[col].median())

# 2. Fill missing listing metadata with 0
metadata_cols = ["product_name_lenght", "product_description_lenght", "product_photos_qty"]
for col in metadata_cols:
    products_cleaned[col] = products_cleaned[col].fillna(0)

# 3. Final null check across all columns
print("Final Missing Values Count:")
print(products_cleaned.isnull().sum())

In [ ]:
# Check for orphan product_ids in order_items
orphan_products = (~order_items["product_id"].isin(products_cleaned["product_id"])).sum()
print("Orphan products in order_items:", orphan_products)

In [ ]:
# Export cleaned dataset
products_cleaned.to_csv("../data/processed/products_cleaned.csv", index=False)
print("Saved successfully to data/processed/products_cleaned.csv")

### Customers Data set

In [ ]:
# 1. Load customers dataset
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")

# 2. Shape and column overview
print("Shape:", customers.shape)
print("\nColumns:", customers.columns.tolist())
customers.info()

# 3. Missing values & Exact duplicate rows
print("\nMissing values:\n", customers.isnull().sum())
print("\nExact duplicate rows:", customers.duplicated().sum())

# 4. Key identification check: customer_id vs customer_unique_id
print("\nUnique customer_id count:", customers["customer_id"].nunique())
print("Unique customer_unique_id count:", customers["customer_unique_id"].nunique())

In [ ]:
# 1. Unique states count aur list
print("Total unique states:", customers["customer_state"].nunique())
print("State distribution:\n", customers["customer_state"].value_counts())

# 2. Check top repeat customers count
orders_per_customer = customers["customer_unique_id"].value_counts()
print("\nMax orders placed by a single customer:", orders_per_customer.max())
print("Number of repeat customers (placed > 1 order):", (orders_per_customer > 1).sum())

In [ ]:
# 1. Clean city names: strip whitespace aur ensure consistency
customers_cleaned = customers.copy()
customers_cleaned["customer_city"] = customers_cleaned["customer_city"].str.strip()

# 2. Check referential integrity with orders
orphan_customers = (~orders["customer_id"].isin(customers_cleaned["customer_id"])).sum()
print("Orphan orders (missing customer record):", orphan_customers)

In [ ]:
customers_cleaned.to_csv("../data/processed/customers_cleaned.csv", index=False)
print("Saved successfully to data/processed/customers_cleaned.csv")

## Reviews data set


In [ ]:
# 1. Load order_reviews dataset
reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")

# 2. Shape, columns, and memory info
print("Shape:", reviews.shape)
print("\nColumns:", reviews.columns.tolist())
reviews.info()

# 3. Missing values check
print("\nMissing values:\n", reviews.isnull().sum())

# 4. Duplicate checks
print("\nExact duplicate rows:", reviews.duplicated().sum())
print("Unique review_id count:", reviews["review_id"].nunique())
print("Unique order_id count:", reviews["order_id"].nunique())

# 5. Review score distribution (1 to 5 stars)
print("\nReview score distribution:\n", reviews["review_score"].value_counts().sort_index())

In [ ]:
# Check duplicate order_ids in reviews
duplicate_orders = reviews[reviews["order_id"].duplicated(keep=False)].sort_values(by="order_id")
print("Total rows involved in duplicate order reviews:", duplicate_orders.shape[0])
duplicate_orders[["review_id", "order_id", "review_score", "review_creation_date", "review_answer_timestamp"]].head(10)

In [ ]:
# 1. Convert review timestamps to datetime
reviews["review_creation_date"] = pd.to_datetime(reviews["review_creation_date"])
reviews["review_answer_timestamp"] = pd.to_datetime(reviews["review_answer_timestamp"])

# 2. Sort by answer timestamp and keep the latest review per order_id
reviews_sorted = reviews.sort_values(by="review_answer_timestamp")
reviews_cleaned = reviews_sorted.drop_duplicates(subset=["order_id"], keep="last").copy()

# 3. Fill missing text columns with empty strings
reviews_cleaned["review_comment_title"] = reviews_cleaned["review_comment_title"].fillna("")
reviews_cleaned["review_comment_message"] = reviews_cleaned["review_comment_message"].fillna("")

# 4. Verification checks
print("Original reviews count:", reviews.shape[0])
print("Cleaned reviews count (unique orders):", reviews_cleaned.shape[0])
print("Remaining duplicate order_ids:", reviews_cleaned["order_id"].duplicated().sum())
print("Null count in cleaned dataset:\n", reviews_cleaned.isnull().sum())

In [ ]:
orphan_reviews = (~reviews_cleaned["order_id"].isin(orders["order_id"])).sum()
print("Orphan reviews (orders not found):", orphan_reviews)

In [ ]:
# Export cleaned reviews
reviews_cleaned.to_csv("../data/processed/order_reviews_cleaned.csv", index=False)
print("Saved successfully to data/processed/order_reviews_cleaned.csv")

### Sellers DataSet

In [ ]:
# 1. Load sellers dataset
sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")

# 2. Shape, columns, and data types
print("Shape:", sellers.shape)
print("\nColumns:", sellers.columns.tolist())
sellers.info()

# 3. Missing values check
print("\nMissing values:\n", sellers.isnull().sum())

# 4. Duplicate checks
print("\nExact duplicate rows:", sellers.duplicated().sum())
print("Unique seller_id count:", sellers["seller_id"].nunique())

In [ ]:
# 1. Check unique states count & distribution
print("Total unique seller states:", sellers["seller_state"].nunique())
print("\nTop 5 seller states:\n", sellers["seller_state"].value_counts().head(5))

# 2. Clean city names (strip whitespace)
sellers_cleaned = sellers.copy()
sellers_cleaned["seller_city"] = sellers_cleaned["seller_city"].str.strip()

# 3. Referential integrity: Kya order_items ke saare sellers is list mein hain?
orphan_sellers = (~order_items["seller_id"].isin(sellers_cleaned["seller_id"])).sum()
print("\nOrphan seller_ids in order_items:", orphan_sellers)

In [ ]:
sellers_cleaned.to_csv("../data/processed/sellers_cleaned.csv", index=False)
print("Saved successfully to data/processed/sellers_cleaned.csv")

### Geolocation DataSet

In [ ]:
# 1. Load geolocation dataset
geo = pd.read_csv("../data/raw/olist_geolocation_dataset.csv")

# 2. Shape, columns, and data types
print("Shape:", geo.shape)
print("\nColumns:", geo.columns.tolist())
geo.info()

# 3. Missing values and duplicates
print("\nMissing values:\n", geo.isnull().sum())
print("\nExact duplicate rows:", geo.duplicated().sum())
print("Unique zip code prefixes:", geo["geolocation_zip_code_prefix"].nunique())

In [ ]:
# 1. Zip code prefix ke basis par lat/lng ka mean aur city/state pick karna
geo_cleaned = geo.groupby("geolocation_zip_code_prefix").agg({
    "geolocation_lat": "mean",
    "geolocation_lng": "mean",
    "geolocation_city": "first",
    "geolocation_state": "first"
}).reset_index()

# 2. Verification checks
print("Original shape:", geo.shape)
print("Aggregated shape:", geo_cleaned.shape)
print("Unique zip code count:", geo_cleaned["geolocation_zip_code_prefix"].nunique())
print("Duplicate zip codes remaining:", geo_cleaned["geolocation_zip_code_prefix"].duplicated().sum())
print("\nNull values in cleaned geo:\n", geo_cleaned.isnull().sum())

In [ ]:
# Check coverage for customers & sellers
cust_missing_geo = (~customers_cleaned["customer_zip_code_prefix"].isin(geo_cleaned["geolocation_zip_code_prefix"])).sum()
seller_missing_geo = (~sellers_cleaned["seller_zip_code_prefix"].isin(geo_cleaned["geolocation_zip_code_prefix"])).sum()

print("Customer orders with missing geo coordinates:", cust_missing_geo, f"({cust_missing_geo / len(customers_cleaned) * 100:.2f}%)")
print("Sellers with missing geo coordinates:", seller_missing_geo, f"({seller_missing_geo / len(sellers_cleaned) * 100:.2f}%)")

In [ ]:
# Export cleaned geolocation
geo_cleaned.to_csv("../data/processed/geolocation_cleaned.csv", index=False)
print("Saved successfully to data/processed/geolocation_cleaned.csv")